# 기프트코 FAQ RAG v2 — 하이브리드 검색 (BM25 + 임베딩)

**Naive RAG(v1) 한계 극복** — 키워드 정밀도 + 의미 검색을 동시에 잡는 Advanced RAG입니다.

| 버전 | 검색 방식 | 강점 | 약점 |
|------|-----------|------|------|
| v0 | TF-IDF | 구현 단순 | 의미 이해 불가 |
| v1 | 임베딩 (Naive RAG) | 표현 달라도 의미 검색 | 정확 키워드 약함 |
| **v2** | **BM25 + 임베딩 (Advanced RAG)** | **키워드 + 의미 동시 커버** | 구현 복잡도 증가 |

## 실행 흐름
1. 데이터 로드 (v1과 동일 FAQ 38개)
2. **BM25 이론** — 수식, k₁/b 파라미터, TF-IDF 비교
3. BM25 인덱스 구축
4. 임베딩 검색 재사용 (v1 캐시 `.npy`)
5. **하이브리드 검색** — RRF(Reciprocal Rank Fusion) 결합
6. v1 vs v2 비교 실험
7. **Elasticsearch 소개** — 실전 하이브리드 검색 엔진
8. LLM 최종 답변 생성

## 0. 패키지 설치

In [ ]:
# 처음 실행 시에만 필요합니다
# !pip install -q rank_bm25 sentence-transformers anthropic scikit-learn pandas openpyxl numpy requests

## 1. 라이브러리 & 데이터 로드

In [ ]:
import os
import numpy as np
import pandas as pd
import requests
from pathlib import Path
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

FAQ_PATH  = Path('../data/기프트코_FAQ_보완_v1.xlsx')
CACHE_DIR = Path('../cache')  # v1이 만든 임베딩 캐시를 기법 이름으로 재사용

raw    = pd.read_excel(FAQ_PATH, dtype=str).fillna('')
rag_df = raw[raw['답변'].str.strip() != ''].copy().reset_index(drop=True)
rag_df['문서ID'] = [f'FAQ-{i+1:04d}' for i in range(len(rag_df))]

print(f'전체: {len(raw)}개  /  RAG 사용: {len(rag_df)}개')
rag_df[['문서ID', '구분', '질문']].head(5)

## 2. BM25 이론 — TF-IDF보다 왜 나은가?

### BM25란?

BM25(Best Matching 25)는 1994년 Robertson & Walker가 제안한 확률적 정보 검색 알고리즘입니다.  
'25'는 25번의 실험적 반복 끝에 도달한 최적 형태라는 의미이며,  
현재 **Elasticsearch, Solr, Lucene의 기본 랭킹 알고리즘**입니다.

---

### 2-1. TF-IDF의 두 가지 구조적 문제

**문제 1: TF 포화(Saturation) 없음**

| 단어 등장 횟수 | TF-IDF 점수 | 실제 정보 증가량 |
|:------------:|:---------:|:--------------:|
| 1번 | 1.0 | 높음 (처음 발견) |
| 2번 | 2.0 | 보통 |
| 10번 | **10.0** | **낮음** (이미 알고 있음) |

→ TF-IDF는 단어가 10번 나오면 점수가 선형으로 10배가 됩니다.  
→ 하지만 "배송"이 10번 나온다고 해서 1번보다 10배 더 관련 있지는 않습니다.

**문제 2: 문서 길이 보정 부족**

짧은 FAQ(50자)에 "배송" 1번 vs 긴 FAQ(500자)에 "배송" 1번  
→ 짧은 문서의 "배송"이 핵심일 가능성이 훨씬 높지만, TF-IDF는 이를 구별하지 못합니다.

---

### 2-2. BM25 수식

$$\text{BM25}(q, d) = \sum_{t \in q} \text{IDF}(t) \cdot \frac{TF(t,d) \cdot (k_1 + 1)}{TF(t,d) + k_1 \cdot \left(1 - b + b \cdot \frac{|d|}{\text{avgdl}}\right)}$$

| 기호 | 의미 | 기본값 |
|------|------|:------:|
| $k_1$ | TF 포화 속도 조절 파라미터 | 1.2 |
| $b$ | 문서 길이 보정 강도 파라미터 | 0.75 |
| $|d|$ | 현재 문서 길이 (토큰 수) | — |
| $\text{avgdl}$ | 전체 문서 평균 길이 | — |
| $\text{IDF}(t)$ | 역문서빈도 (희귀 단어일수록 높음) | — |

---

### 2-3. k₁ 파라미터: TF 포화(Saturation)

분자에 `(k₁ + 1)`이, 분모에 `TF + k₁`이 있어서 TF가 아무리 커져도 점수가 `(k₁+1)/1 = k₁+1`에 수렴합니다.

```
TF값   :   1      2      3      5     10     20     50
TF-IDF :  1.0    2.0    3.0    5.0   10.0   20.0   50.0   (선형 무한 증가)
BM25   :  0.92   0.96   0.97   0.98   0.99   1.00   1.00   (k1=1.2, b=0 가정, IDF=1)
         ↑ 첫 등장에 정보 집중           ↑ 반복은 거의 차이 없음
```

- **k₁ = 0**: TF 무시 → 단어 존재 여부만 봄 (binary 검색)
- **k₁ = 1.2** (기본값): 적당한 포화 → 반복에 덜 민감
- **k₁ → ∞**: TF-IDF와 동일하게 선형 증가

---

### 2-4. b 파라미터: 문서 길이 정규화

분모의 `(1 - b + b × |d|/avgdl)` 항이 문서 길이에 따라 TF를 보정합니다.

- **b = 0**: 문서 길이 완전 무시 → 모든 문서 동등 취급
- **b = 0.75** (기본값): 부분 정규화 → 짧은 문서의 키워드에 약간 더 가중치
- **b = 1**: 완전 정규화 → 모든 문서를 평균 길이로 환산 후 비교

> **FAQ 데이터 특성**: 문서 길이가 비슷하므로 `avgdl`에 가까워 b의 영향이 작습니다.  
> 웹 문서처럼 길이 편차가 크면 b = 0.75가 중요한 역할을 합니다.

---

### 2-5. 검색 방식 비교 요약

| 방식 | 키워드 정밀도 | 의미 이해 | 표현 다양성 | 속도 | 대표 도구 |
|------|:-----------:|:-------:|:---------:|:----:|----------|
| TF-IDF | ★★★ | ✗ | ✗ | ⚡ | sklearn |
| **BM25** | **★★★★** | ✗ | ✗ | ⚡ | rank_bm25, Elasticsearch |
| 임베딩 | ★★ | ✓ | ✓ | 🐢 | sentence-transformers |
| **하이브리드 (v2)** | **★★★★** | **✓** | **✓** | ⚡🐢 | **이 노트북** |

In [ ]:
# BM25 TF 포화 효과 — 수치로 직접 확인
k1 = 1.2

print('TF 횟수가 늘어날 때 BM25 vs TF-IDF 점수 변화 (k1=1.2, b=0 가정, IDF=1 고정)')
print(f'{"TF 횟수":>8}  {"TF-IDF":>10}  {"BM25":>10}  {"증가율(BM25)":>12}  {"시각화"}')
print('-' * 65)

prev_bm25 = 0
for tf in [1, 2, 3, 5, 10, 20, 50]:
    tfidf = float(tf)
    bm25_score = (tf * (k1 + 1)) / (tf + k1)
    delta = bm25_score - prev_bm25
    bar = '█' * max(1, int(bm25_score * 10))
    print(f'{tf:>8}  {tfidf:>10.2f}  {bm25_score:>10.4f}  {delta:>+12.4f}  {bar}')
    prev_bm25 = bm25_score

print()
print('→ TF-IDF는 선형 무한 증가, BM25는 2.2에 수렴 — 반복 단어에 휘둘리지 않음')
print(f'  BM25 수렴 상한값: k1 + 1 = {k1 + 1}')

## 3. BM25 인덱스 구축

`rank_bm25` 라이브러리의 `BM25Okapi` 클래스를 사용합니다.  
입력은 **토큰화된 문서 리스트** — 각 문서는 토큰(단어)의 리스트입니다.

### 한국어 토크나이저 전략

| 방식 | 정확도 | 추가 설치 | 추천 상황 |
|------|:------:|:--------:|----------|
| 공백 분리 (이 노트북) | ★★★ | 없음 | 프로토타입, FAQ 같은 짧은 문서 |
| 형태소 분석 (konlpy/Mecab) | ★★★★★ | 필요 | 프로덕션, 긴 문서 |
| 문자 n-gram | ★★★★ | 없음 | 오타 허용, 부분 매칭 |

> FAQ는 짧고 정형화된 문장이므로 공백 분리만으로도 충분히 동작합니다.

In [ ]:
def korean_tokenize(text):
    # 공백 기반 분리 — 형태소 분석기(konlpy) 없이도 FAQ 수준에서 잘 동작
    return text.strip().split()

def make_bm25_doc(row):
    # 구분 + 질문 + 답변 모두 포함 → 키워드 커버리지 최대화
    return row['구분'] + ' ' + row['질문'] + ' ' + row['답변']

bm25_corpus = [korean_tokenize(make_bm25_doc(row)) for _, row in rag_df.iterrows()]
bm25 = BM25Okapi(bm25_corpus, k1=1.2, b=0.75)

print('BM25 인덱스 생성 완료')
print(f'  문서 수   : {len(bm25_corpus)}개')
print(f'  k1        : {bm25.k1}')
print(f'  b         : {bm25.b}')
print(f'  평균 문서 길이: {bm25.avgdl:.1f} 토큰')
print(f'  어휘 크기 : {len(bm25.idf)}개 고유 토큰')

# 예시: '시안' 토큰의 IDF 확인
print()
for word in ['시안', '배송', '결제', '가능', '주문']:
    idf = bm25.idf.get(word, 0)
    print(f'  IDF("{word}"): {idf:.4f}')
print('→ IDF가 높을수록 전체 문서에서 희귀한 단어 (= 더 변별력 있음)')

In [ ]:
def bm25_search(query, top_k=3):
    tokens = korean_tokenize(query)
    scores = bm25.get_scores(tokens)
    top_idx = scores.argsort()[::-1][:top_k]
    result = rag_df.iloc[top_idx].copy()
    result.insert(0, 'score', scores[top_idx])
    return result[['score', '문서ID', '구분', '질문']].reset_index(drop=True)

# 검색 테스트
test_queries = [
    '시안 수정은 몇 번이나 가능한가요?',
    '세금계산서 발행',
    '배송 얼마나 걸리나요?',
    '제작 취소할 수 있나요?',
    '파일 어떻게 보내나요?',
]

print('=== BM25 검색 결과 ===\n')
for q in test_queries:
    top1 = bm25_search(q, top_k=1).iloc[0]
    print(f'질문: {q}')
    print(f'  BM25 1위: [{top1["구분"]}] {top1["질문"]}  (score={top1["score"]:.4f})')
print()
print('※ BM25는 키워드 정확 매칭에 강합니다 — 특히 "세금계산서" 같은 전문 용어에 효과적')

## 4. 임베딩 검색 재사용 (v1 캐시)

v1에서 생성한 `faq_embeddings.npy` 파일을 그대로 재사용합니다.  
모델 재학습 없이 임베딩 캐시를 로드하므로 **시간과 비용 절약**이 가능합니다.

In [ ]:
MODEL_NAME  = 'jhgan/ko-sroberta-multitask'
EMBED_PATH  = CACHE_DIR / 'faq_embeddings_jhgan.npy'

print(f'모델 로딩: {MODEL_NAME}')
model = SentenceTransformer(MODEL_NAME)
print(f'임베딩 차원: {model.get_embedding_dimension()}')

if EMBED_PATH.exists():
    faq_embeddings = np.load(EMBED_PATH)
    print(f'캐시 로드: {EMBED_PATH}  shape={faq_embeddings.shape}')
else:
    print('캐시 없음 — 새로 생성합니다...')
    faq_embeddings = model.encode(rag_df['질문'].tolist(), normalize_embeddings=True, show_progress_bar=True)
    np.save(EMBED_PATH, faq_embeddings)
    print(f'저장 완료: {EMBED_PATH}')

def embed_search(query, top_k=3):
    q_vec  = model.encode([query], normalize_embeddings=True)
    scores = np.dot(faq_embeddings, q_vec.T).flatten()
    top_idx = scores.argsort()[::-1][:top_k]
    result  = rag_df.iloc[top_idx].copy()
    result.insert(0, 'score', scores[top_idx])
    return result[['score', '문서ID', '구분', '질문', '답변']].reset_index(drop=True)

# 빠른 확인
q = '시안 수정은 몇 번이나 가능한가요?'
print(f'\n임베딩 검색 (v1 방식): {q}')
for _, r in embed_search(q, top_k=3).iterrows():
    print(f'  score={r["score"]:.4f}  [{r["구분"]}] {r["질문"]}')

## 5. 하이브리드 검색 — RRF(Reciprocal Rank Fusion)

### 왜 단순 점수 합산이 안 될까?

```
BM25 점수  : 0.0 ~ 20.0   (문서/쿼리마다 스케일이 크게 다름)
임베딩 점수 : 0.0 ~  1.0   (정규화된 코사인 유사도)
```

단순 합산하면 BM25 점수가 임베딩을 압도합니다.  
Min-Max 정규화를 해도 두 분포의 특성이 달라 공정한 비교가 어렵습니다.

---

### RRF: 점수(score) 대신 순위(rank)를 합산

$$\text{RRF}(d) = \sum_{r \in R} \frac{1}{k + \text{rank}_r(d)}$$

- $k$ : 상위 랭크 편중 방지 상수 (보통 **60**)
- $\text{rank}_r(d)$ : 검색 방식 $r$에서 문서 $d$의 순위 (1부터 시작)

---

### RRF 수치 예시 (k=60)

```
문서        BM25 순위   임베딩 순위   RRF 점수
────────────────────────────────────────────────
FAQ-0001       1위         5위     1/61 + 1/65 = 0.0320  ← BM25만 잘함
FAQ-0027       5위         1위     1/65 + 1/61 = 0.0320  ← 임베딩만 잘함
FAQ-0016       2위         2위     1/62 + 1/62 = 0.0323  ← 두 방식 모두 동의 → 1위!
```

**핵심**: 두 검색 방식이 **동시에 상위**에 올린 문서가 최종 1위가 됩니다.  
한 방식에서만 1위여도, 다른 방식에서 낮으면 최종 순위가 분산됩니다 → **앙상블 효과**

---

### RRF vs 다른 결합 방식 비교

| 방식 | 원리 | 장점 | 단점 |
|------|------|------|------|
| 단순 합산 | score_bm25 + score_emb | 직관적 | 스케일 불균형 문제 |
| 가중 합산 | α·bm25 + (1-α)·emb | 비율 조절 가능 | α 튜닝 필요 |
| **RRF** | **순위 기반 합산** | **스케일 무관, 파라미터 적음** | 점수 정보 손실 |

In [ ]:
def rrf_fusion(ranked_lists, k=60):
    # ranked_lists: 각 검색 방식의 문서 인덱스 순위 리스트들
    scores = {}
    for ranked in ranked_lists:
        for rank, doc_idx in enumerate(ranked):
            scores[doc_idx] = scores.get(doc_idx, 0.0) + 1.0 / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

def hybrid_search(query, top_k=3, rrf_k=60):
    # 1. BM25 순위
    bm25_scores  = bm25.get_scores(korean_tokenize(query))
    bm25_ranking = list(bm25_scores.argsort()[::-1])

    # 2. 임베딩 순위
    q_vec        = model.encode([query], normalize_embeddings=True)
    embed_scores = np.dot(faq_embeddings, q_vec.T).flatten()
    embed_ranking = list(embed_scores.argsort()[::-1])

    # 3. RRF 결합
    fused      = rrf_fusion([bm25_ranking, embed_ranking], k=rrf_k)
    top_idx    = np.array([idx for idx, _ in fused[:top_k]])
    rrf_scores = [score for _, score in fused[:top_k]]

    result = rag_df.iloc[top_idx].copy()
    result.insert(0, 'rrf_score',   rrf_scores)
    result.insert(1, 'bm25_score',  bm25_scores[top_idx])
    result.insert(2, 'embed_score', embed_scores[top_idx])
    return result[['rrf_score', 'bm25_score', 'embed_score', '문서ID', '구분', '질문', '답변']].reset_index(drop=True)

print('hybrid_search 함수 준비 완료')
print('  rrf_score   : RRF 최종 점수 (이 값으로 최종 순위 결정)')
print('  bm25_score  : BM25 원점수 (참고용)')
print('  embed_score : 임베딩 코사인 유사도 (참고용)')

In [ ]:
def print_hybrid_result(query, top_k=3):
    res = hybrid_search(query, top_k)
    print(f'질문: {query}')
    print('=' * 75)
    for _, r in res.iterrows():
        print(f'  RRF={r["rrf_score"]:.5f}  BM25={r["bm25_score"]:.3f}  Emb={r["embed_score"]:.3f}')
        print(f'  [{r["구분"]}] {r["질문"]}')
    print()

# 하이브리드 검색 테스트
test_queries = [
    '시안 수정은 몇 번이나 가능한가요?',
    '디자인 변경 부탁드려요',
    '배송 얼마나 걸리나요?',
    '세금계산서 부탁드려요',
]

for q in test_queries:
    print_hybrid_result(q)

## 6. v1(임베딩) vs v2(하이브리드) 비교 실험

v1의 실패 케이스를 중심으로 v2가 어떻게 개선되는지 확인합니다.

**핵심 실패 케이스**  
"시안 수정은 몇 번이나 가능한가요?" 질문에서  
- v1(임베딩): 제작기간 FAQ를 1위로 반환 ← 오답
- BM25: "시안", "수정" 키워드가 있는 FAQ-0016을 정확히 찾음
- v2(하이브리드): BM25의 키워드 정밀도 + 임베딩의 의미 검색을 결합 → 개선

In [ ]:
compare_queries = [
    ('시안 수정은 몇 번이나 가능한가요?',   'FAQ-0016'),  # 시안 수정 3회 제한
    ('디자인 바꿀 수 있어요?',             'FAQ-0016'),  # 의미 검색 필요
    ('배송 얼마나 걸리나요?',              'FAQ-0008'),  # 배송 관련
    ('파일 어떻게 보내나요?',              'FAQ-0032'),  # 인쇄 파일 전달
    ('세금계산서 발행 부탁해요',            'FAQ-0036'),  # 정확 키워드
    ('영수증 다시 받을 수 있나요?',         'FAQ-0037'),  # 의미 + 키워드
]

print(f'{"질문":<24}  {"v1 임베딩 1위":<30}  {"BM25 1위":<30}  {"v2 하이브리드 1위":<30}  {"정답"}')
print('─' * 130)

emb_correct = 0
hyb_correct = 0

for query, ans_id in compare_queries:
    emb_top = embed_search(query, top_k=1).iloc[0]
    bm_top  = bm25_search(query, top_k=1).iloc[0]
    hyb_top = hybrid_search(query, top_k=1).iloc[0]

    emb_str = f'[{emb_top["구분"]}] {emb_top["질문"][:20]}...' if len(emb_top['질문']) > 20 else f'[{emb_top["구분"]}] {emb_top["질문"]}'
    bm_str  = f'[{bm_top["구분"]}] {bm_top["질문"][:20]}...'  if len(bm_top['질문']) > 20  else f'[{bm_top["구분"]}] {bm_top["질문"]}'
    hyb_str = f'[{hyb_top["구분"]}] {hyb_top["질문"][:20]}...' if len(hyb_top['질문']) > 20 else f'[{hyb_top["구분"]}] {hyb_top["질문"]}'

    emb_ok = '✓' if emb_top['문서ID'] == ans_id else '✗'
    hyb_ok = '✓' if hyb_top['문서ID'] == ans_id else '✗'
    if emb_top['문서ID'] == ans_id: emb_correct += 1
    if hyb_top['문서ID'] == ans_id: hyb_correct += 1

    short_q = query[:22] + '..' if len(query) > 22 else query
    print(f'{short_q:<24}  {emb_str:<30}  {bm_str:<30}  {hyb_str:<30}  {emb_ok}→{hyb_ok}')

print()
print(f'v1 임베딩 정답률: {emb_correct}/{len(compare_queries)}')
print(f'v2 하이브리드 정답률: {hyb_correct}/{len(compare_queries)}')

## 7. Elasticsearch 소개 — 프로덕션 하이브리드 검색 엔진

지금까지 구현한 BM25 + 임베딩 하이브리드 검색을 **실제 서비스**에 올리려면  
Elasticsearch(ES)가 업계 표준 선택입니다.

---

### 7-1. Elasticsearch란?

Apache Lucene 기반의 오픈소스 분산 검색 엔진 (2010년 Shay Banon 출시)

```
핵심 특징:
  ✅ BM25를 기본 스코어링 알고리즘으로 채택
  ✅ 8.x 버전부터 kNN 벡터 검색 네이티브 지원
  ✅ 8.9 버전부터 RRF 내장 (별도 구현 불필요)
  ✅ REST API — 어떤 언어에서도 HTTP로 호출
  ✅ 실시간 색인 — 문서 추가/삭제 즉시 검색 반영
  ✅ 수억 건 문서도 밀리초 단위 검색
  ✅ 클러스터 분산 처리 — 수평 확장 가능
```

---

### 7-2. 우리 프로토타입 vs Elasticsearch

| 항목 | 이 노트북 | Elasticsearch |
|------|:--------:|:-------------:|
| 최대 문서 수 | ~수만 건 | 수억 건 |
| 실시간 업데이트 | 전체 재인덱싱 | 즉시 추가/삭제 |
| 하이브리드 검색 | 수동 RRF 구현 | 내장 RRF (8.9+) |
| 한국어 처리 | 공백 분리 | Nori 형태소 분석기 내장 |
| 운영 | 단일 프로세스 | 클러스터 분산 |
| 모니터링 | 없음 | Kibana 대시보드 |
| 적합 규모 | **프로토타입, 소규모** | **스타트업 ~ 대기업** |

---

### 7-3. Elasticsearch 아키텍처

```
[클라이언트 (Python/JS/curl)]
         ↓  REST API (JSON over HTTP)
[Elasticsearch 클러스터]
   ├── Node 1 (마스터 + 데이터)
   ├── Node 2 (데이터)
   └── Node 3 (데이터)
        ├── Index: giftco_faq
        │    ├── Shard 0  (BM25 역인덱스 + Dense Vector)
        │    └── Shard 1
        └── Index: giftco_products   ← 나중에 추가 가능
```

---

### 7-4. 핵심 개념 매핑

| Elasticsearch 용어 | 관계형 DB 비유 | 이 노트북 대응 |
|:----------------:|:------------:|:------------:|
| Index | 테이블 | `rag_df` + `faq_embeddings` |
| Document | 행(Row) | FAQ 항목 1개 |
| Field | 컬럼 | `질문`, `답변`, `구분` |
| Mapping | 스키마 | dtype 정의 |
| Analyzer | — | `korean_tokenize()` |
| dense_vector | — | `faq_embeddings.npy` |
| Shard | 파티션 | — (단일 프로세스) |

---

### 7-5. 로컬 실행 방법

```bash
# Docker로 단일 노드 실행 (가장 빠른 시작)
docker run -p 9200:9200 \
  -e "discovery.type=single-node" \
  -e "xpack.security.enabled=false" \
  elasticsearch:8.12.0

# 동작 확인
curl http://localhost:9200
```

클라우드 옵션: [Elastic Cloud](https://cloud.elastic.co) 14일 무료 체험 / AWS OpenSearch

In [ ]:
# ── Elasticsearch 코드 예시 ─────────────────────────────────────────────
# 실행 조건:
#   pip install elasticsearch
#   docker run -p 9200:9200 -e 'discovery.type=single-node' elasticsearch:8.12.0
#
# 아래 코드는 개념 이해용입니다. 서버 없이도 읽을 수 있습니다.
# ────────────────────────────────────────────────────────────────────────

ES_CODE = '''
from elasticsearch import Elasticsearch

es = Elasticsearch('http://localhost:9200')

# 1. 인덱스 생성 — BM25 파라미터 + 벡터 필드 동시 설정
es.indices.create(
    index='giftco_faq',
    body={
        'settings': {
            'similarity': {
                'bm25_custom': {'type': 'BM25', 'k1': 1.2, 'b': 0.75}
            },
            'analysis': {
                'analyzer': {
                    'korean': {'type': 'custom', 'tokenizer': 'nori_tokenizer'}  # Nori 형태소 분석기
                }
            }
        },
        'mappings': {
            'properties': {
                'doc_id':    {'type': 'keyword'},
                'category':  {'type': 'keyword'},
                'question':  {'type': 'text', 'analyzer': 'korean', 'similarity': 'bm25_custom'},
                'answer':    {'type': 'text', 'analyzer': 'korean'},
                'embedding': {                   # 벡터 검색 필드
                    'type': 'dense_vector',
                    'dims': 768,
                    'index': True,
                    'similarity': 'cosine'
                }
            }
        }
    }
)

# 2. 문서 색인 (BM25 텍스트 + 임베딩 동시 저장)
for _, row in rag_df.iterrows():
    q_vec = model.encode(row["질문"], normalize_embeddings=True)
    es.index(
        index='giftco_faq',
        id=row['문서ID'],
        body={
            'doc_id':    row['문서ID'],
            'category':  row['구분'],
            'question':  row['질문'],
            'answer':    row['답변'],
            'embedding': q_vec.tolist()
        }
    )

# 3. 하이브리드 검색 — Elasticsearch 8.9+ 내장 RRF
query = '시안 수정은 몇 번 가능한가요?'
q_vec = model.encode([query], normalize_embeddings=True)[0].tolist()

results = es.search(
    index='giftco_faq',
    body={
        'retriever': {
            'rrf': {                               # ES 내장 RRF!
                'retrievers': [
                    {
                        'standard': {              # BM25 키워드 검색
                            'query': {'match': {'question': query}}
                        }
                    },
                    {
                        'knn': {                   # 벡터 검색
                            'field': 'embedding',
                            'query_vector': q_vec,
                            'k': 5,
                            'num_candidates': 20
                        }
                    }
                ],
                'rank_constant': 60,               # RRF k 파라미터
                'window_size': 10
            }
        },
        'size': 3
    }
)

for hit in results['hits']['hits']:
    print(f"  [{hit['_source']['category']}] {hit['_source']['question']}")
    print(f"  _score={hit['_score']:.4f}  id={hit['_id']}")
'''

print('=== Elasticsearch 하이브리드 검색 코드 (개념 예시) ===')
print(ES_CODE)
print('실행하려면: pip install elasticsearch && Docker로 ES 서버 구동 후 위 코드 실행')

## 8. LLM 최종 답변 생성

하이브리드 검색으로 더 정확하게 찾은 FAQ를 컨텍스트로 LLM 답변을 생성합니다.

```
사용자 질문
   → hybrid_search()로 관련 FAQ top-k 검색 (BM25 + 임베딩 RRF)
   → FAQ 내용을 프롬프트에 삽입
   → LLM이 고객 응대 말투로 최종 답변 생성
```

**우선순위**: Ollama(로컬) → Anthropic API → OpenRouter

In [ ]:
OLLAMA_BASE_URL    = 'http://localhost:11434'
OLLAMA_MODEL       = 'qwen2.5:3b'
ANTHROPIC_API_KEY  = os.getenv('ANTHROPIC_API_KEY', '')
OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY', '')

def check_ollama():
    try:
        r = requests.get(f'{OLLAMA_BASE_URL}/api/tags', timeout=3)
        models = [m['name'] for m in r.json().get('models', [])]
        print(f'Ollama 실행 중  설치된 모델: {models}')
        return True
    except Exception:
        print('Ollama 미실행 (ollama serve 또는 앱 실행 필요)')
        return False

OLLAMA_AVAILABLE = check_ollama()
print()
print('사용할 LLM:',
      f'Ollama ({OLLAMA_MODEL})' if OLLAMA_AVAILABLE else
      'Anthropic API'            if ANTHROPIC_API_KEY  else
      'OpenRouter'               if OPENROUTER_API_KEY else
      '없음 (프롬프트만 출력)')

def build_prompt_v2(query, retrieved_df):
    blocks = []
    for i, (_, row) in enumerate(retrieved_df.iterrows()):
        header = '[근거 ' + str(i+1) + '] 구분: ' + row['구분'] + ' / FAQ 질문: ' + row['질문']
        body   = 'FAQ 답변: ' + row['답변']
        blocks.extend([header, body, ''])
    context = '\n'.join(blocks)
    lines = [
        '당신은 기프트코 고객지원 FAQ 챗봇입니다.',
        '아래 FAQ 근거만 사용해서 답변하세요.',
        'FAQ에 없는 내용은 단정하지 말고 고객센터 문의를 안내하세요.',
        '',
        '[고객 질문]',
        query,
        '',
        '[검색된 FAQ 근거]',
        context,
        '[답변 작성 기준]',
        '- 결론을 먼저 말하세요.',
        '- FAQ 근거 내용만 사용하세요.',
        '- 마지막에 참고 문서ID를 표시하세요.',
    ]
    return '\n'.join(lines)

In [ ]:
def generate_answer_v2(query, top_k=3):
    retrieved = hybrid_search(query, top_k=top_k)
    prompt    = build_prompt_v2(query, retrieved)
    system    = '당신은 제공된 근거만 사용해서 답변하는 한국어 고객지원 챗봇입니다.'

    if OLLAMA_AVAILABLE:
        resp = requests.post(
            f'{OLLAMA_BASE_URL}/api/chat',
            json={
                'model':    OLLAMA_MODEL,
                'messages': [
                    {'role': 'system', 'content': system},
                    {'role': 'user',   'content': prompt},
                ],
                'options': {'temperature': 0.2},
                'stream':  False,
            },
            timeout=120,
        )
        resp.raise_for_status()
        answer = resp.json()['message']['content']

    elif ANTHROPIC_API_KEY:
        import anthropic
        client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
        msg    = client.messages.create(
            model='claude-haiku-4-5-20251001',
            max_tokens=1024,
            system=system,
            messages=[{'role': 'user', 'content': prompt}],
        )
        answer = msg.content[0].text

    elif OPENROUTER_API_KEY:
        resp = requests.post(
            'https://openrouter.ai/api/v1/chat/completions',
            headers={'Authorization': f'Bearer {OPENROUTER_API_KEY}'},
            json={
                'model': 'openai/gpt-4o-mini',
                'messages': [
                    {'role': 'system', 'content': system},
                    {'role': 'user',   'content': prompt},
                ],
                'temperature': 0.2,
            },
            timeout=60,
        )
        resp.raise_for_status()
        answer = resp.json()['choices'][0]['message']['content']

    else:
        print('LLM 미설정 — 프롬프트 출력:')
        print('=' * 60)
        print(prompt)
        return {'answer': None, 'retrieved': retrieved, 'prompt': prompt}

    return {'answer': answer, 'retrieved': retrieved, 'prompt': prompt}

In [ ]:
# 하이브리드 RAG 최종 테스트
test_questions = [
    '시안 수정은 몇 번이나 가능한가요?',
    '디자인 바꿔달라고 하면 어떻게 하나요?',
    '배송 현황은 어디서 확인하나요?',
]

for q in test_questions:
    result = generate_answer_v2(q, top_k=3)

    print(f'Q: {q}')
    print('─' * 70)

    if result['answer']:
        print(f'A: {result["answer"]}')
        print()
        print('참고된 FAQ (하이브리드 검색 결과):')
        for _, r in result['retrieved'].iterrows():
            print(f'  {r["문서ID"]}  RRF={r["rrf_score"]:.5f}  BM25={r["bm25_score"]:.3f}  Emb={r["embed_score"]:.3f}  [{r["구분"]}] {r["질문"]}')
    print()
    print('=' * 70)
    print()

## 다음 단계

### 이 노트북에서 배운 것

| 개념 | 핵심 포인트 |
|------|------------|
| BM25 | TF 포화(k₁)와 문서 길이 정규화(b)로 TF-IDF의 두 한계를 극복 |
| RRF | 점수 스케일 문제를 순위 기반 합산으로 우회 — 파라미터 적고 강건함 |
| 하이브리드 | 키워드 검색(BM25)의 정밀도 + 임베딩의 의미 이해를 앙상블 |
| Elasticsearch | BM25 내장 + kNN + RRF 지원 — 프로토타입 → 프로덕션 전환점 |

---

### RAG v3에서 더 발전시킬 것들

| 개선 포인트 | 방법 | 난이도 |
|------------|------|:------:|
| **검색 정확도** | Cross-Encoder 리랭킹 | ★★★ |
| **한국어 BM25** | konlpy(Mecab) 형태소 분석 | ★★ |
| **쿼리 확장** | 동의어/유의어 사전 적용 | ★★ |
| **지식 확장** | 상품 데이터 + FAQ 통합 검색 | ★★★ |
| **자동 평가** | Ragas 또는 LLM-as-Judge | ★★★★ |
| **스케일업** | Elasticsearch 실전 배포 | ★★★★ |

---

### RAG 발전 단계 로드맵

```
v0  TF-IDF RAG          → 단순 키워드 매칭
 ↓
v1  Naive RAG (임베딩)   → 의미 검색 추가
 ↓
v2  Advanced RAG (현재)  → BM25 + 임베딩 하이브리드 + RRF
 ↓
v3  Advanced RAG+       → Cross-Encoder 리랭킹 + 쿼리 리라이팅
 ↓
v4  Agentic RAG         → 반복 검색 + 도구 선택 + Self-RAG
```